# 서울시 인구 분석

<img src='https://raw.githubusercontent.com/Jangrae/img/master/people2.png' width="650" align="left">

## 1. 라이브러리 불러오기

- 사용할 라이브러리를 불러옵니다.

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import minmax_scale

* 데이터 불러오기

    - 다음 경로의 파일을 읽어와 pop01, pop02, pop03 데이터프레임을 만듭니다.
    - 파일 경로1: https://raw.githubusercontent.com/Jangrae/csv/master/seoul_pop_h01.csv
    - 파일 경로2: https://raw.githubusercontent.com/Jangrae/csv/master/seoul_pop_h02.csv
    - 파일 경로3: https://raw.githubusercontent.com/Jangrae/csv/master/seoul_pop_h03.csv

In [6]:
pop01=pd.read_csv("https://raw.githubusercontent.com/Jangrae/csv/master/seoul_pop_h01.csv")

In [7]:
pop02=pd.read_csv("https://raw.githubusercontent.com/Jangrae/csv/master/seoul_pop_h02.csv")

In [8]:
pop03=pd.read_csv("https://raw.githubusercontent.com/Jangrae/csv/master/seoul_pop_h03.csv")

## 2. 데이터 탐색

다음과 같은 정보 확인을 통해 처리할 대상 데이터를 이해합니다.
- 상/하위 데이터 확인
- 데이터프레임 크기 확인
- 열 이름, 데이터 형식, 값 개수 등 확인
- 기초 통계정보 확인
- 결측치 확인
- 범주형 데이터 확인
- 개별 열 값 상세 확인 등

**1) 데이터프레임 크기 확인**

- 세 개의 데이터프레임 크기(행 수, 열 수)를 확인합니다.

In [10]:
pop01.shape

(40, 3)

In [11]:
pop02.shape

(36, 3)

In [12]:
pop03.shape

(40, 3)

**2) year 최솟값, 최댓값 확인**

- 세 개의 데이터프레임 year열 최솟값, 최댓값 크기를 비교해 차이가 있는 지 각각 확인합니다.

In [15]:
print(pop01.year.max(),pop01.year.min())
print(pop02.year.max(),pop02.year.min())
print(pop03.year.max(),pop03.year.min())

2020 1981
2020 1985
2020 1981


**3) 결측치 확인**

- 세 개의 데이터프레임에 결측치가 있는 지 각각 확인합니다.

In [16]:
pop01.isna().sum()

year        0
k_male      0
k_female    0
dtype: int64

In [18]:
pop02.isna().sum()

year        0
f_male      0
f_female    0
dtype: int64

In [19]:
pop03.isna().sum()

year         0
household    0
older_65     0
dtype: int64

**4) 열 정보 확인**

- 세 개의 데이터프레임의 열 정보를 확인합니다.

In [20]:
pop01.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   year      40 non-null     int64
 1   k_male    40 non-null     int64
 2   k_female  40 non-null     int64
dtypes: int64(3)
memory usage: 1.1 KB


In [21]:
pop02.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   year      36 non-null     int64
 1   f_male    36 non-null     int64
 2   f_female  36 non-null     int64
dtypes: int64(3)
memory usage: 992.0 bytes


In [22]:
pop03.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   year       40 non-null     int64
 1   household  40 non-null     int64
 2   older_65   40 non-null     int64
dtypes: int64(3)
memory usage: 1.1 KB


## 3. 데이터 전처리

전처리 과정에서 다음과 같은 처리를 할 수 있습니다.

- 결측치 처리
- 값 변경
- 열 추가
- 불필요한 열 제거
- 열 이름 변경
- 데이터 통합(연결 또는 조인)
- 가변수화 등

**1) 데이터 통합**

- concat을 사용해 연결하면 인덱스 값을 기준으로 연결되어 데이터가 어긋납니다.
- **year** 열을 기준으로 **outer** 방식으로 조인(병합)합니다.
- 외국인 정보가 1981 ~ 1984년이 누락되어 결측치가 발생할 것입니다.
- 세 개의 데이터프레임을 병합(조인)하여 pop 데이터프레임을 선언합니다.
- 이후의 모든 작업은 pop 데이터프레임을 대상으로 진행합니다.

In [26]:
mid=pd.merge(pop01,pop02,how="outer",on="year")

In [29]:
pop=pd.merge(mid,pop03,how="outer",on="year")

In [30]:
pop.head()

,year,k_male,k_female,f_male,f_female,household,older_65
0,1981,4160,4191,NaN,NaN,1915,246
1,1982,4160,4191,NaN,NaN,2001,260
2,1983,4160,4191,NaN,NaN,2116,260
3,1984,4160,4191,NaN,NaN,2246,275
4,1985,4160,4191,7.0,6.0,2338,211


**2) 결측치 확인**

- 결측치가 있는지 확인합니다.

In [31]:
pop.isna().sum()

year         0
k_male       0
k_female     0
f_male       4
f_female     4
household    0
older_65     0
dtype: int64

**3) 결측치 처리**

- 연도별 인구 현황이므로 임의의 값을 채우는 것이 바람직하지 않아 보입니다.
- 이후 값, 즉 1985년 값으로 채우는 것도 정확한 분석을 방해할 것 같습니다.
- 이에 결측치가 있는 1981년~1984년 행을 제거할 것입니다.

In [36]:
pop=pop[pop.year>1985]

In [38]:
pop.head()

,year,k_male,k_female,f_male,f_female,household,older_65
5,1986,4899,4888,7.0,5.0,2428,305
6,1987,5000,4979,6.0,5.0,2518,329
7,1988,5156,5120,5.0,5.0,2658,349
8,1989,5305,5261,6.0,5.0,2817,363
9,1990,5321,5282,5.0,4.0,2820,363


**3) 열 추가**

- 이후 분석의 편의를 위해 다음과 같은 의미를 갖는 열을 추가하고자 합니다.
    - k_total = 전체 한국인 인구수
    - f_total = 전체 외국인 인구수
    - male = 전체 남자 인구수
    - female = 전체 여자 인구수
    - total = 전체 인구수
- 추가할 열에 대한 공식은 다음과 같습니다.
    - k_total = k_male + k_female
    - f_total = f_male + f_female
    - male = k_male + f_male
    - female = k_female + f_female
    - total = k_total + f_total
- 정리한 공식에 따라 데이터프레임에 열을 추가합니다.

In [44]:
pop.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 35 entries, 5 to 39
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   year       35 non-null     int64  
 1   k_male     35 non-null     int64  
 2   k_female   35 non-null     int64  
 3   f_male     35 non-null     float64
 4   f_female   35 non-null     float64
 5   household  35 non-null     int64  
 6   older_65   35 non-null     int64  
 7   k_total    35 non-null     int64  
dtypes: float64(2), int64(6)
memory usage: 3.5 KB


**4) 열 순서 변경**

- 데이터 이해를 돕기 위해 다음과 같은 순서로 데이터프레임 열 순서를 변경합니다.
- year, household, total, male, female, k_total, k_male, k_female, f_total, f_male, f_female, older_65


**5) 인덱스 초기화**

- 인덱스가 0부터 시작하는 일련 변호를 갖지 않는다면 인덱스를 초기화합니다.

## 4. 추가 전처리

* x, y 구분

    - x : feature
    - y : target <- total

* train test split

* Scaling

    - min-max scaling

## 5. 모델링

* 모델 선언

* 모델 학습

* 학습한 모델 기반으로 예측값 생성

## 6. 평가